# SU(2) symmetry + TrigEncoder — automated hypothesis sweep

Explores whether **spin-coupling** on trig Fourier feature legs (harmonic doublets) helps classification vs dense baseline and bucket-flux U(1).

| Hypothesis | What we test |
|------------|--------------|
| **H0** | TrigEncoder + Gram baseline beats chance on drift data |
| **H1** | Bucket-flux U(1) (torus phase proxy) has measurable compat |
| **H2** | SU(2) feature coupling preserves symmetry (`residual≈0`) with sparser allowed blocks |
| **H3** | SU(2) hard vs soft vs baseline — accuracy vs `allowed_frac` trade-off |
| **H4** | `angle_mode` (`:discrete_cell_pi` vs `:dense_torus`) shifts Φ-weighted charge separation |
| **H5** | Higher `D_max` recovers SU(2) capacity (hard mask at 2.6% allowed is too aggressive at D=32) |

Re-run **§1** after pulling library changes (`su2_symmetry.jl`).


In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "../.."))
Pkg.instantiate()

using MPSFast
using MPSFast.Encoders
using Random, Statistics, Printf
using Plots

function refresh_mpsfast_training!()
    pkg = pkgdir(MPSFast)
    src = joinpath(pkg, "src")
    # Reload only experiment-hot files (avoids doc-replacement warning spam).
    for f in ("su2_symmetry.jl", "training.jl", "core.jl")
        Base.include(MPSFast, joinpath(src, f))
    end
    global train_mps! = MPSFast.train_mps!
    global cosine_lr = MPSFast.cosine_lr
end
refresh_mpsfast_training!()

D_max, η, ε_cut = 32, 5e-4, 1e-5
n_epochs, patience = 30, 5
N_train, N_test, M = 2500, 500, 12
EXPERIMENT_SEEDS = [7, 13, 21, 31, 42]  # 5 seeds for lower variance
seed_main = EXPERIMENT_SEEDS[1]
compat_gate = 0.15
_pct(x) = 100 * x

# Vector{Any} panel rows break `unique(f, itr)` — use explicit helpers.
_panel_modes(panel) = [m for m in unique(r.mode for r in panel)]
_safe_mean(f, xs; default=NaN) = isempty(xs) ? default : mean(f, xs)
_safe_std(f, xs) = length(xs) < 2 ? 0.0 : std(map(f, xs))
D_for_features(d) = d <= 4 ? D_max : max(D_max, 64)  # d=6 needs wider bonds under SU(2)


## 1. Data & TrigEncoder bundles


In [ ]:
function synthetic_paths_labels(N, M; drift=0.15, σ=1.0, rng=Random.default_rng())
    paths = zeros(Float64, N, M)
    labels = Vector{Int}(undef, N)
    for i in 1:N
        y = rand(rng, 1:2)
        d = y == 1 ? drift : -drift
        paths[i, :] = cumsum(d .+ σ .* randn(rng, M))
        labels[i] = y
    end
    return paths, labels
end

function make_trig_bundle(enc::TrigEncoder; rng=Random.default_rng())
    tr_p, tr_y = synthetic_paths_labels(N_train, M; rng=rng)
    te_p, te_y = synthetic_paths_labels(N_test, M; rng=rng)
    fit_grid!(enc, tr_p)
    xi_tr = encode_labeled_paths(enc, tr_p, tr_y; n_classes=2)
    xi_te = encode_labeled_paths(enc, te_p, te_y; n_classes=2)
    Phi = Float32.(feature_map(enc))
    K = 2^enc.m
    Ml = classification_chain_length(enc, M)
    d = site_dim(enc)
    return (; enc, Phi, K, Ml, d, xi_tr, xi_te, tr_y, te_y, tr_p, te_p)
end

rng0 = MersenneTwister(seed_main)
enc_main = TrigEncoder(3, 4; angle_mode=:discrete_cell_pi)
data = make_trig_bundle(enc_main; rng=rng0)
@printf("Trig bundle: K=%d  d_feat=%d  Ml=%d  train=%s  test=%s\n",
    data.K, data.d, data.Ml, string(size(data.xi_tr)), string(size(data.xi_te)))


## 2. Phase A — Φ structure & charge diagnostics

Each harmonic pair `(cos hθ, sin hθ)` is a spin-`j=h/2` doublet. Bucket row `k` induces a **Φ-weighted** effective `m_z` charge.


In [ ]:
function phi_charge_table(data)
    Phi = data.Phi
    rows = []
    for k in 1:data.K
        q = trig_phi_weighted_charge(k, Phi)
        push!(rows, (bucket=k, q_eff=round(q; digits=3)))
    end
    return rows
end

function class_charge_separation(data)
    Phi = data.Phi
    qs = Float64[]
    ys = Int[]
    for (xi_row, y) in zip(eachrow(data.xi_tr), data.tr_y)
        qpath = sum(trig_phi_weighted_charge(xi_row[t], Phi) for t in 1:(data.Ml - 1))
        push!(qs, qpath)
        push!(ys, y)
    end
    m1 = mean(qs[ys .== 1])
    m2 = mean(qs[ys .== 2])
    return (; m1, m2, gap=abs(m1 - m2))
end

sep = class_charge_separation(data)
@printf("Class-conditional Σ q_eff: y=1 → %.2f   y=2 → %.2f   gap=%.2f\n", sep.m1, sep.m2, sep.gap)

bucket_spec = su2_trig_empirical_bucket_spec(data.K, 2, data.xi_tr)
compat_bucket = su2_bucket_compat(data.xi_tr, bucket_spec, data.K, 2)
@printf("Bucket-flux compat (empirical C4): %.3f\n", compat_bucket)


## 3. Phase B — automated mode sweep

Modes: `baseline`, `u1_bucket`, `su2_feature`, `combo` (U1 label flux + SU2 path features).


In [ ]:
const MODES = (
    baseline = (; u1=false, su2=false, su2_cons=:none),
    u1_bucket = (; u1=true, su2=false, su2_cons=:none),
    su2_hard = (; u1=false, su2=true, su2_cons=:hard),
    su2_soft = (; u1=false, su2=true, su2_cons=:soft),
    combo_hard = (; u1=true, su2=true, su2_cons=:hard),
)

function train_trig_mode(data; mode::Symbol, seed::Int, n_ep=n_epochs, D=nothing, return_mps=false)
    cfg = MODES[mode]
    rng = MersenneTwister(seed)
    Ml, d, Phi = data.Ml, data.d, data.Phi
    D_eff = D === nothing ? D_for_features(d) : D
    η_eff = d <= 4 ? η : η / 2  # gentler updates for d=6 feature maps
    bucket = su2_trig_empirical_bucket_spec(data.K, 2, data.xi_tr)
    compat_b = su2_bucket_compat(data.xi_tr, bucket, data.K, 2)
    u1 = cfg.u1 ? su2_as_u1_bucket_spec(bucket, data.K, 2) : NoSymmetry
    su2 = cfg.su2 ? su2_trig_feature_spec(d, 2) : NoSU2Symmetry
    use_u1 = symmetry_active(u1) && compat_b >= compat_gate
    mps = if su2_active(su2)
        init_mps_classification_su2(Ml, d, 2, D_eff, su2; rng=rng)
    elseif use_u1
        init_mps_classification_u1(Ml, d, 2, D_eff, u1; rng=rng)
    else
        init_mps_classification(Ml, d, 2, D_eff; rng=rng)
    end
    frac0 = su2_active(su2) ? su2_allowed_fraction(mps, su2; n_classes=2, d_path=d) : 1.0
    train_mps!(mps, data.xi_tr, n_ep, η_eff, D_eff, ε_cut;
        feature_phi=Phi, verbose=false, nll_samples=300,
        val_data=data.xi_te, val_samples=200, patience=patience,
        u1_spec=use_u1 ? u1 : NoSymmetry,
        u1_conservation=use_u1 ? :hard : :none,
        su2_spec=su2_active(su2) ? su2 : NoSU2Symmetry,
        su2_conservation=cfg.su2_cons,
        su2_soft_strength=0.85,
        n_classes=2, d_path=d)
    acc = classification_accuracy(mps, data.xi_te, 2; phi=Phi)
    allowed = su2_active(su2) ? su2_allowed_fraction(mps, su2; n_classes=2, d_path=d) : 1.0
    residual = su2_active(su2) ? su2_symmetry_residual(mps, su2; n_classes=2, d_path=d) : 0.0
    row = (; mode, seed, acc, allowed, allowed_init=frac0, residual,
            compat_bucket=compat_b, use_u1)
    return return_mps ? merge(row, (; mps=mps)) : row
end

function run_mode_panel(data; seeds=EXPERIMENT_SEEDS)
    results = NamedTuple[]
    for mode in keys(MODES)
        for s in seeds
            r = train_trig_mode(data; mode=mode, seed=s)
            push!(results, r)
            @printf("%-12s seed=%d  acc=%5.1f%%  allow=%.3f (init %.3f)  res=%.2e  u1_on=%s\n",
                string(mode), s, _pct(r.acc), r.allowed, r.allowed_init, r.residual, r.use_u1)
        end
    end
    return results
end

panel_B = run_mode_panel(data)


## 4. Phase C — encoder geometry sweep (`angle_mode` × `d_features`)


In [ ]:
function sweep_geometry(; seeds=EXPERIMENT_SEEDS)
    configs = [
        (; m=3, d=4, angle=:discrete_cell_pi),
        (; m=3, d=4, angle=:dense_torus),
        (; m=3, d=6, angle=:discrete_cell_pi),
    ]
    rows = NamedTuple[]
    for (i, c) in enumerate(configs)
        enc = TrigEncoder(c.m, c.d; angle_mode=c.angle)
        dset = make_trig_bundle(enc; rng=MersenneTwister(100 + i))
        sep = class_charge_separation(dset)
        for mode in (:baseline, :su2_hard, :su2_soft)
            accs = Float64[]
            for s in seeds
                r = train_trig_mode(dset; mode=mode, seed=s, n_ep=20)
                push!(accs, r.acc)
            end
            μ = _safe_mean(x -> x, accs)
            σ = _safe_std(x -> x, accs)
            push!(rows, (; config=string(c.angle, "_d", c.d), mode,
                acc_mean=μ, acc_std=σ, charge_gap=sep.gap))
            @printf("geom %-22s %-12s  acc=%5.1f±%.1f%%  charge_gap=%.2f\n",
                rows[end].config, mode, _pct(μ), _pct(σ), rows[end].charge_gap)
        end
    end
    return rows
end

geom_rows = sweep_geometry()


## 5b. Capacity sweep — `D_max` vs SU(2) hard `allowed_frac`


In [ ]:
function capacity_sweep(data; D_list=[32, 48, 64], seed=7)
    for D in D_list
        su2 = su2_trig_feature_spec(data.d, 2)
        mps = init_mps_classification_su2(data.Ml, data.d, 2, D, su2; rng=MersenneTwister(seed))
        frac = su2_allowed_fraction(mps, su2; n_classes=2, d_path=data.d)
        r = train_trig_mode(data; mode=:su2_hard, seed=seed, n_ep=15, D=D)
        @printf("D_max=%2d  allow_init=%.3f  acc=%5.1f%%\n", D, frac, _pct(r.acc))
    end
end

capacity_sweep(data)


## 5. Phase D — robustness: global rescale `S → αS` (eval only)


In [ ]:
function eval_rescale(data, mps; α=1.5)
    enc = data.enc
    paths = data.te_p .* α
    xi = encode_labeled_paths(enc, paths, data.te_y; n_classes=2)
    classification_accuracy(mps, xi, 2; phi=data.Phi)
end

function robustness_panel(data; seed=EXPERIMENT_SEEDS[2])
    rows = NamedTuple[]
    for mode in (:baseline, :su2_soft)
        r = train_trig_mode(data; mode=mode, seed=seed, return_mps=true)
        acc0 = classification_accuracy(r.mps, data.xi_te, 2; phi=data.Phi)
        paths = data.te_p .* 1.5
        xi = encode_labeled_paths(data.enc, paths, data.te_y; n_classes=2)
        acc1 = classification_accuracy(r.mps, xi, 2; phi=data.Phi)
        push!(rows, (; mode, acc0, acc1, delta=acc1 - acc0, allowed=r.allowed))
        @printf("rescale %-12s  clean=%5.1f%%  α=1.5 → %5.1f%%  Δ=%+.1f pp  allow=%.3f\n",
            mode, _pct(acc0), _pct(acc1), _pct(rows[end].delta), r.allowed)
    end
    return rows
end

robust_rows = robustness_panel(data)


## 6. Auto-summary — flag interesting outcomes


In [ ]:
function summarize_panel(panel)
    isempty(panel) && error("panel is empty — re-run Phase B (cell above) first")
    modes = _panel_modes(panel)
    println("\n=== Phase B summary (mean test acc) ===")
    best_mode, best_acc = :baseline, 0.0
    for m in modes
        sub = [r for r in panel if r.mode == m]
        μ = _safe_mean(r -> r.acc, sub)
        σ = _safe_std(r -> r.acc, sub)
        @printf("%-12s  %5.1f ± %.1f%%  (n=%d)\n", m, _pct(μ), _pct(σ), length(sub))
        if μ > best_acc
            best_acc, best_mode = μ, m
        end
    end
    baseline_sub = [r for r in panel if r.mode == :baseline]
    su2_soft_sub = [r for r in panel if r.mode == :su2_soft]
    su2_hard_sub = [r for r in panel if r.mode == :su2_hard]
    baseline_μ = _safe_mean(r -> r.acc, baseline_sub; default=0.5)
    su2_μ = _safe_mean(r -> r.acc, su2_soft_sub; default=0.5)
    hard_frac = _safe_mean(r -> r.allowed_init, su2_hard_sub; default=0.0)
    println("\n--- Hypothesis flags ---")
    println("H0 (baseline > 55%): ", baseline_μ > 0.55 ? "✓ PASS" : "✗ inconclusive")
    h2_ok = !isempty(su2_hard_sub) && all(r -> r.residual < 1e-5, su2_hard_sub)
    println("H2 (SU2 hard residual≈0): ", h2_ok ? "✓ PASS" : "✗ check")
    @printf("H5 (SU2 hard allowed_frac at D=%d): %.1f%% → %s\n", D_max, 100 * hard_frac,
        hard_frac < 0.1 ? "capacity collapse — try D_max↑ or :soft" : "ok")
    delta = su2_μ - baseline_μ
    println(@sprintf("H3 (SU2 soft vs baseline): Δacc = %+.1f pp → %s",
        _pct(delta), abs(delta) > 0.02 ? "INTERESTING" : "small effect"))
    println("Best mode: ", best_mode, " (", round(_pct(best_acc); digits=1), "%)")
    return (; best_mode, best_acc, delta)
end

summary = summarize_panel(panel_B)

let modes = collect(keys(MODES)),
    accs = [_safe_mean(x -> x.acc, [r for r in panel_B if r.mode == m]; default=0.5) for m in modes]
    bar(string.(modes), _pct.(accs); ylab="test accuracy (%)", title="Phase B: symmetry modes",
        legend=false, xrotation=30)
end
